# Base SDLX

In [13]:
import torch
from diffusers import AutoPipelineForText2Image
from diffusers.utils import load_image

# 1. Load an initial image (Change this to a local path or keep the URL)
# The model works best if the image is resized to 1024x1024
init_image = load_image("Cat03.jpg").convert("RGB")
init_image = init_image.resize((1024, 1024))

# 2. Initialize the Pipeline (SDXL)
# We use variant="fp16" to save ~10GB of disk space and VRAM
model_id = "stabilityai/stable-diffusion-xl-base-1.0"
pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16
).to("cuda")
pipe.load_lora_weights(
    "lora_styles/coke_product_lora.safetensors",
    adapter_name="product"
)

pipe.set_adapters("product", adapter_weights=0.85)


# 3. Optimization for 12GB VRAM
# This is the "magic" line for 4070 users. It ensures you never hit an OOM error.
pipe.enable_model_cpu_offload()

# 4. Run Generation
# 'strength' is the most important parameter: 
# 0.0 = original image, 1.0 = completely new image. 0.6 is a good middle ground.
prompt = "a product photo of a front view sks_product red soda bottle, white background"
image = pipe(
    prompt=prompt, 
).images[0]

# 5. Save Output
image.save("img2img_result.png")
print("✓ Generation complete! Check 'img2img_result.png' in your folder.")

100%|██████████| 50/50 [00:18<00:00,  2.74it/s]


✓ Generation complete! Check 'img2img_result.png' in your folder.


In [24]:
pipe.set_adapters("product", adapter_weights=0.75)

#prompt = "a cinematic advertisement photo of a sks_product red soda bottle floating in outer space, dramatic rim lighting, stars in the background, realistic perspective"
prompt = "a cyberpunk neon advertisement of a sks_product red soda bottle, close up view, futuristic city lights, purple glow, realistic perspective, cinematic reflections"

negative_prompt = "blurry, distorted, text, watermark"
image = pipe(
    prompt=prompt,
    num_inference_steps=50,
    guidance_scale=12.0, # Увеличь, чтобы стиль проявился ярче
    target_size=(1024, 1024),
    original_size=(1024, 1024)
).images[0]


prompts = [
    "photo of sks product",
    "sks product on table",
    "sks product in hand",
    "3D render of sks product",
    "sketch of sks product"
]
# 5. Save Output
image.save("img2img_result.png")
print("✓ Generation complete! Check 'img2img_result.png' in your folder.")

100%|██████████| 50/50 [00:18<00:00,  2.74it/s]


✓ Generation complete! Check 'img2img_result.png' in your folder.


In [9]:
import torch
from diffusers import AutoPipelineForText2Image
from diffusers.utils import load_image

pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
)

pipe.enable_xformers_memory_efficient_attention()

pipe.load_ip_adapter("h94/IP-Adapter", weight_name="ip-adapter_sdxl.bin")
pipe.set_ip_adapter_scale(0.6)

image_prompt = load_image("product_example.jpeg")
prompt = "a premium advertisement banner, realistic lighting, minimalistic style"
neg = "low quality, distorted, blurry, watermark"

with torch.autocast("cuda"):
    result = pipe(
        prompt=prompt,
        ip_adapter_image=image_prompt,
        negative_prompt=neg,
        height=512,
        width=512,
        num_inference_steps=50,
    ).images[0]

result.save("output_ip_adapter_mem_safe.png")

Loading pipeline components...: 100%|██████████| 7/7 [00:14<00:00,  2.07s/it]


NotImplementedError: No operator found for `memory_efficient_attention_forward` with inputs:
     query       : shape=(1, 2, 1, 40) (torch.float32)
     key         : shape=(1, 2, 1, 40) (torch.float32)
     value       : shape=(1, 2, 1, 40) (torch.float32)
     attn_bias   : <class 'NoneType'>
     p           : 0.0
`fa3F@0.0.0` is not supported because:
    xFormers wasn't build with CUDA support
    dtype=torch.float32 (supported: {torch.bfloat16, torch.float16})
    operator wasn't built - see `python -m xformers.info` for more info
`fa2F@2.5.7-pt` is not supported because:
    xFormers wasn't build with CUDA support
    dtype=torch.float32 (supported: {torch.bfloat16, torch.float16})
`cutlassF-pt` is not supported because:
    xFormers wasn't build with CUDA support

# ControlNet

In [5]:
import torch
from diffusers import StableDiffusionXLControlNetPipeline, ControlNetModel
from diffusers.utils import load_image
from PIL import Image
import cv2
import numpy as np

# 1. Load ControlNet (Canny) - The "Structural Guard"
controlnet = ControlNetModel.from_pretrained(
    "diffusers/controlnet-canny-sdxl-1.0", 
    torch_dtype=torch.float16
).to("cuda")

# 2. Load SDXL Pipeline
pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    controlnet=controlnet,
    torch_dtype=torch.float16,
    variant="fp16"
).to("cuda")

lora_path = "lora_styles/luxury_lora.safetensors"
pipe.load_lora_weights(lora_path)
# 4. Memory Optimizations for 12GB VRAM
pipe.enable_model_cpu_offload() 
#pipe.enable_xformers_memory_efficient_attention()

# 5. Process the Product Image
product_img = load_image("coke.jpg").resize((1024, 1024))

# Generate Canny Edges
image_np = np.array(product_img)
# Adjust 100, 200 to change how many details of the product are kept
edges = cv2.Canny(image_np, 100, 200) 
edges = edges[:, :, None]
edges = np.concatenate([edges, edges, edges], axis=2)
canny_image = Image.fromarray(edges)

# 6. Define the "Style" via Prompt
# Use keywords your LoRA was trained on (e.g., 'mrt_style')
prompt = "a professional luxury_product_advertisement for a product, mrt_style, high-end photography, 8k"
negative_prompt = "text, watermark, blurry, low quality, distorted product, messy background, colorful noise"

# 7. Generate
image = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    image=canny_image,
    controlnet_conditioning_scale=0.8, # 0.7-0.8 is the sweet spot for product ads
    num_inference_steps=35,
    guidance_scale=7,
).images[0]

image.save("text_to_banner_result_no_lora.png")
print("Banner generated using Text + ControlNet!")

KeyboardInterrupt: 

# ControlNet + IP-adapter

In [ ]:
import torch
from diffusers import StableDiffusionXLControlNetPipeline, ControlNetModel
from transformers import CLIPVisionModelWithProjection
from diffusers.utils import load_image
from PIL import Image
import cv2
import numpy as np


## IP-Adapter + ControlNet

image_encoder = CLIPVisionModelWithProjection.from_pretrained(
    "laion/CLIP-ViT-H-14-laion2B-s32B-b79K",
    torch_dtype=torch.float16
).to("cuda")

controlnet = ControlNetModel.from_pretrained(
    "diffusers/controlnet-canny-sdxl-1.0", 
    torch_dtype=torch.float16
).to("cuda")

pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    controlnet=controlnet,
    image_encoder=image_encoder,
    torch_dtype=torch.float16,
    variant="fp16"
).to("cuda")

pipe.load_ip_adapter(
    "h94/IP-Adapter", 
    subfolder="sdxl_models", 
    weight_name="ip-adapter-plus_sdxl_vit-h.safetensors"
)

pipe.enable_model_cpu_offload()
pipe.enable_xformers_memory_efficient_attention()

product_img = load_image("https://i.pinimg.com/736x/bf/74/e7/bf74e7ce2057395dd85a98d83ea83aea.jpg").convert("RGB").resize((1024, 1024))

image_np = np.array(product_img)
edges = cv2.Canny(image_np, 100, 200)
edges = edges[:, :, None]
edges = np.concatenate([edges, edges, edges], axis=2)
canny_image = Image.fromarray(edges)

pipe.set_ip_adapter_scale(0.7)

prompt = "a professional advertisement banner, high-end product photography, minimalist studio, soft lighting, 8k"
negative_prompt = "gold, yellow, orange, deformed, messy, blurry, low quality"

image = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    image=canny_image,           # Держит форму
    ip_adapter_image=product_img, # Держит цвет (синий)
    controlnet_conditioning_scale=0.8,
    num_inference_steps=30,
    guidance_scale=7.5,
).images[0]

image.save("final_blue_banner.png")
image.show()

Loading weights: 100%|██████████| 520/520 [00:00<00:00, 3849.74it/s]
CLIPVisionModelWithProjection LOAD REPORT from: laion/CLIP-ViT-H-14-laion2B-s32B-b79K
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...23}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...23}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...23}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...23}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...23}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...23}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...23}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...23}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...23}.self_attn.

OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 11.57 GiB of which 2.12 MiB is free. Including non-PyTorch memory, this process has 11.46 GiB memory in use. Of the allocated memory 10.89 GiB is allocated by PyTorch, and 410.73 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [3]:
import torch
from diffusers import StableDiffusionXLControlNetPipeline, ControlNetModel
from diffusers.utils import load_image
from PIL import Image
import cv2
import numpy as np

# 1. Load ControlNet (Canny) - The "Structural Guard"
controlnet = ControlNetModel.from_pretrained(
    "diffusers/controlnet-canny-sdxl-1.0", 
    torch_dtype=torch.float16
).to("cuda")

# 2. Load SDXL Pipeline
pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    controlnet=controlnet,
    torch_dtype=torch.float16,
    variant="fp16"
).to("cuda")

# 3. Load your Fine-Tuned Style LoRA
# Replace with the path to your .safetensors file
# If you haven't trained one yet, you can comment this out to use base SDXL
pipe.load_lora_weights("path/to/your/apple_style_lora.safetensors")

# 4. Memory Optimizations for 12GB VRAM
pipe.enable_model_cpu_offload() 
pipe.enable_xformers_memory_efficient_attention()

# 5. Process the Product Image
product_img = load_image("product_example.jpg").resize((1024, 1024))

# Generate Canny Edges
image_np = np.array(product_img)
# Adjust 100, 200 to change how many details of the product are kept
edges = cv2.Canny(image_np, 100, 200) 
edges = edges[:, :, None]
edges = np.concatenate([edges, edges, edges], axis=2)
canny_image = Image.fromarray(edges)

# 6. Define the "Style" via Prompt
# Use keywords your LoRA was trained on (e.g., 'mrt_style')
prompt = "a professional advertisement banner for a product, mrt_style, minimalist white studio, soft cinematic shadows, high-end photography, 8k"
negative_prompt = "text, watermark, blurry, low quality, distorted product, messy background, colorful noise"

# 7. Generate
image = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    image=canny_image,
    controlnet_conditioning_scale=0.7, # 0.7-0.8 is the sweet spot for product ads
    num_inference_steps=35,
    guidance_scale=7.5,
).images[0]

image.save("text_to_banner_result.png")
print("Banner generated using Text + ControlNet!")

/home/marat/Desktop/CourseWork/xformers_env/lib/python3.11/site-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


OutOfMemoryError: CUDA out of memory. Tried to allocate 14.00 MiB. GPU 0 has a total capacity of 11.57 GiB of which 2.12 MiB is free. Including non-PyTorch memory, this process has 11.46 GiB memory in use. Of the allocated memory 10.94 GiB is allocated by PyTorch, and 356.63 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)